Every month, auto loan securitization trusts file loan-level data with the SEC under
Form ABS-EE: balance, payment status, days past due, vehicle make, borrower credit score —
for every individual loan in the pool. It's public. It's on EDGAR. Most people
don't parse it.

In April 2026 the filed universe covers **5.4 million active loans** across 171 trusts.
That's a lot of car debt.

This note looks at one slice of it: who's borrowing, what their credit profile looked
like at origination, and how much the car was worth relative to the loan. Three numbers —
FICO, LTV, and credit tier — tell most of the story.

- **FICO** — obligor credit score at origination (when the lender was still optimistic)
- **LTV** — original loan amount ÷ vehicle value at origination
- **Credit tier** — FICO buckets: prime+, prime, near-prime, subprime, deep sub

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE  = "#4A6FA5"
AMBER = "#C8964A"
RED   = "#C86A50"
TEAL  = "#5A8A8A"
GREEN = "#7EA870"
GREY  = "#D4CFC4"
BG    = "#FAF9F5"
TEXT  = "#1C1C1E"
MUTED = "#8A8A8A"

TIER_COLORS = {
    "750+  Prime+":        BLUE,
    "700-749  Prime":      TEAL,
    "660-699  Near-Prime": GREEN,
    "620-659  Subprime":   AMBER,
    "<620  Deep Sub":      RED,
}

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   BG,
    "axes.edgecolor":   MUTED,
    "axes.labelcolor":  TEXT,
    "xtick.color":      TEXT,
    "ytick.color":      TEXT,
    "text.color":       TEXT,
    "font.family":      "sans-serif",
    "font.size":        11,
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

PERIOD = "2026-04"
OUTPUTS = Path("../../analysis/outputs")

by_deal = pd.read_parquet(OUTPUTS / f"fico_by_deal_{PERIOD}.parquet")
by_make = pd.read_parquet(OUTPUTS / f"fico_by_make_{PERIOD}.parquet")
by_tier = pd.read_parquet(OUTPUTS / f"fico_by_tier_{PERIOD}.parquet")

## Credit tier mix

About a third of the filed pool is prime-plus (FICO 750+). The deep subprime shelf —
below 620 — is nearly as large.

This is what a cross-section of the US securitized auto market looks like. It's not
a subprime story, and it's not a prime story. It's both, in the same dataset, filed
to the same regulator, available for free on EDGAR.

In [ ]:
tier_totals = (
    by_tier.groupby("tier")[["loan_count", "orig_bal_mm"]]
    .sum()
    .reset_index()
    .sort_values("tier")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor=BG)
fig.suptitle(f"ABS-EE Autoloan Credit Tier Mix — {PERIOD}", fontsize=13, color=TEXT, y=1.01)

colors = [TIER_COLORS.get(t, GREY) for t in tier_totals["tier"]]

axes[0].barh(tier_totals["tier"], tier_totals["loan_count"] / 1e6, color=colors)
axes[0].set_xlabel("Loans (millions)")
axes[0].set_title("By loan count")

axes[1].barh(tier_totals["tier"], tier_totals["orig_bal_mm"] / 1e3, color=colors)
axes[1].set_xlabel("Original balance ($B)")
axes[1].set_title("By original balance")

plt.tight_layout()
plt.show()

## When the loan is bigger than the car

Prime borrowers, on average, borrow about 90 cents for every dollar of vehicle value.
Deep subprime borrowers borrow $1.16 for every dollar of vehicle value — on day one,
before the car has left the lot.

That gap matters when things go wrong. Repossess a car from a prime borrower and the
proceeds likely cover the outstanding balance. Repossess one from a deep subprime
borrower and you're eating a loss before the first payment was ever missed.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=BG)

tier_ltv = (
    by_tier.groupby("tier")["avg_ltv"]
    .mean()
    .reset_index()
    .sort_values("tier")
)

colors = [TIER_COLORS.get(t, GREY) for t in tier_ltv["tier"]]
bars = ax.barh(tier_ltv["tier"], tier_ltv["avg_ltv"], color=colors)
ax.axvline(1.0, color=MUTED, linestyle="--", linewidth=1, label="LTV = 1.0  (loan = vehicle value)")
ax.set_xlabel("Average LTV at origination")
ax.set_title(f"LTV by credit tier — {PERIOD}")
ax.legend()

for bar, val in zip(bars, tier_ltv["avg_ltv"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9, color=TEXT)

plt.tight_layout()
plt.show()

## The primest shelves in the room

Captive finance arms lend to the customers buying their own cars. GM Financial
(median FICO 786, avg LTV 0.93), Nissan (797, 0.90), BMW (801, 0.83) — these are
employed people with decent credit, buying a car they want, at a dealership motivated
to close the sale. Good loans, almost by construction.

Bridgecrest (FICO 567, LTV 1.53) is a different business. The average Bridgecrest
loan at origination is worth 53% more than the underlying vehicle. That's not a
mistake — it's how subprime auto securitization works. The yield has to compensate
for something.

In [ ]:
top_deals = by_deal.nlargest(20, "loan_count").sort_values("median_fico")

fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG)

scatter = ax.scatter(
    top_deals["median_fico"],
    top_deals["avg_ltv"],
    s=top_deals["loan_count"] / 800,
    c=top_deals["median_fico"],
    cmap="RdYlGn",
    alpha=0.85,
    edgecolors=MUTED,
    linewidths=0.5,
)

for _, row in top_deals.iterrows():
    label = row["sponsor"][:22] if isinstance(row["sponsor"], str) else ""
    ax.annotate(label, (row["median_fico"], row["avg_ltv"]),
                fontsize=7.5, color=TEXT,
                xytext=(4, 2), textcoords="offset points")

ax.axhline(1.0, color=MUTED, linestyle="--", linewidth=1, alpha=0.6)
ax.set_xlabel("Median FICO")
ax.set_ylabel("Average LTV")
ax.set_title(f"Top 20 deals by loan count — FICO vs LTV — {PERIOD}\n(bubble size = loan count)")
plt.colorbar(scatter, ax=ax, label="Median FICO")
plt.tight_layout()
plt.show()

## The car doesn't know who's driving it

The same Toyota Camry gets financed by a FICO-780 Capital One Prime customer
and a FICO-580 Exeter customer. Luxury brands cluster at the top — the overlap
between "people who buy new BMWs" and "people with 800 FICOs" is real, and not
particularly surprising.

The interesting cases are the volume brands with wide distributions: same make,
very different borrower. At that point the brand tells you almost nothing
about credit quality — what matters is who underwrote it.

In [ ]:
top_makes = (
    by_make.groupby("make")
    .agg(loan_count=("loan_count", "sum"), median_fico=("median_fico", "mean"), avg_ltv=("avg_ltv", "mean"))
    .reset_index()
    .nlargest(20, "loan_count")
    .sort_values("median_fico", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor=BG)

norm = plt.Normalize(top_makes["median_fico"].min(), top_makes["median_fico"].max())
colors = plt.cm.RdYlGn(norm(top_makes["median_fico"]))

ax.barh(top_makes["make"], top_makes["median_fico"], color=colors)
ax.set_xlabel("Median FICO at origination")
ax.set_title(f"Top 20 vehicle makes by loan count — median FICO — {PERIOD}")
ax.set_xlim(550, 820)

for _, row in top_makes.iterrows():
    ax.text(row["median_fico"] + 1, row["make"],
            f"{row['median_fico']:.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

---

## Notes on the data

- **FICO missing rate**: ~3.5% of loans in this period have no credit score on file —
  excluded from all charts above. The SEC doesn't require FICO reporting under ABS-EE;
  trusts report it voluntarily. Make of that what you will.
- **Schema variation**: Different trusts file slightly different field sets. Handled
  with a `union_by_name` merge across Parquet files; fields a given trust doesn't
  report show up as null for that trust's loans.
- **FICO timing**: The score is from origination — when the lender was still optimistic.
  Loans that have been in the pool for two years may look quite different today.
- **Source**: SEC EDGAR Form ABS-EE, Schedule AL — autoloan asset class.
  Reporting period ending 2026-03-31, filed April 2026.